# 🔍 Hybrid Multi-Domain Image Forensics — Google Colab
**AI-assisted manipulation detection & localization** (spatial + frequency domain features fused by a dual-stream CNN).

**Before you start:** `Runtime → Change runtime type → T4 GPU` → Save.

Run the cells top to bottom. Total time on a T4 ≈ 1 hour (dataset ~25 min, training ~30 min, evaluation ~5 min).

## 0. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'No GPU - enable it in Runtime > Change runtime type'

## 1. Get the project code
Upload **`HybridImageForensics.zip`** when the file picker appears.
*(Alternative: put the zip in Google Drive and set `ZIP_IN_DRIVE` below.)*

In [ ]:
import os, zipfile
ZIP_IN_DRIVE = None   # e.g. '/content/drive/MyDrive/HybridImageForensics.zip'

if not os.path.exists('/content/HybridImageForensics/forensics'):
    if ZIP_IN_DRIVE:
        from google.colab import drive; drive.mount('/content/drive')
        zipfile.ZipFile(ZIP_IN_DRIVE).extractall('/content')
    else:
        from google.colab import files
        up = files.upload()
        zipfile.ZipFile(next(iter(up))).extractall('/content')
%cd /content/HybridImageForensics
!ls

## 2. Install the extra libraries (everything else is pre-installed on Colab)

In [ ]:
!pip -q install PyWavelets streamlit

## 3. (Optional) Save checkpoints & results to Google Drive
Colab deletes files when the runtime stops. Set `SAVE_TO_DRIVE = True` to keep a copy.

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_DIR = '/content/drive/MyDrive/HybridImageForensics_outputs'
if SAVE_TO_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)

## 4. Generate the training dataset
Authentic images + 5 manipulation types (**splice, copy-move, inpainting / object removal, AI-style generative fill, retouch**), each with a pixel-accurate ground-truth mask.

**Better accuracy on real photos:** upload a folder of your own photos (or any photo dataset) and pass `--real-dir /content/my_photos` – they are then used as the base images instead of procedurally generated scenes.

In [ ]:
REAL_DIR = None   # e.g. '/content/my_photos'  (folder of authentic .jpg/.png photos)
extra = f'--real-dir {REAL_DIR}' if REAL_DIR else ''
!python scripts/generate_dataset.py --train 6000 --val 600 --test 1000 --workers 4 {extra}

## 5. Look at a few training samples (image | ground-truth mask)

In [ ]:
import csv, matplotlib.pyplot as plt, matplotlib.image as mpimg
rows = list(csv.DictReader(open('data/synth/train/labels.csv')))
picks = [r for r in rows if r['label'] == '1'][:5] + [r for r in rows if r['label'] == '0'][:1]
fig, ax = plt.subplots(2, 6, figsize=(20, 7))
for k, r in enumerate(picks):
    ax[0, k].imshow(mpimg.imread('data/synth/train/images/' + r['file'])); ax[0, k].set_title(r['op'])
    if r['mask']: ax[1, k].imshow(mpimg.imread('data/synth/train/masks/' + r['mask']), cmap='gray')
    ax[0, k].axis('off'); ax[1, k].axis('off')
plt.show()

## 6. Train the hybrid model on the GPU
* `--cache` pre-computes the spatial / frequency forensic maps once (Colab has only 2 CPU cores – without it the GPU would wait for the CPU).
* `--amp` = mixed precision (faster on T4).
* The best epoch (by validation AUC + pixel-F1) is saved to `checkpoints/hybrid.pt`.

In [ ]:
!python -u scripts/train.py --streams hybrid --epochs 30 --batch 32 --lr 2e-3 --workers 2 --cache /content/cache --amp

## 7. Training curves

In [ ]:
import json
h = json.load(open('checkpoints/hybrid.history.json'))
ep = [r['epoch'] for r in h]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(ep, [r['train_loss'] for r in h]); ax[0].set_title('train loss')
ax[1].plot(ep, [r['val_auc'] for r in h], label='AUC'); ax[1].plot(ep, [r['val_accuracy'] for r in h], label='accuracy')
ax[1].plot(ep, [r['val_f1'] for r in h], label='F1'); ax[1].legend(); ax[1].set_title('image-level (val)')
ax[2].plot(ep, [r['val_pixel_f1'] for r in h]); ax[2].set_title('pixel-level F1 (val)')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.savefig('results_training_curves.png', dpi=120); plt.show()

## 8. Evaluate on the held-out test set

In [ ]:
!python scripts/evaluate.py --ckpt checkpoints/hybrid.pt --device cuda
from IPython.display import Image as IPImage, display
for f in ['results/hybrid_roc.png', 'results/hybrid_confusion.png']:
    display(IPImage(f, width=420))
display(IPImage('results/hybrid_examples.png', width=700))

## 9. Compare with baselines (and optional ablations)
* **ELA heuristic** – single spatial cue, no learning
* **Hand-crafted multi-domain features + Random Forest** – classic machine learning
* **Ablations** (optional, ~2 × 25 min): the same CNN with only the RGB stream / only the forensic stream – shows the benefit of fusing domains.

In [ ]:
!python scripts/baseline.py --max-train 3000 --workers 4

In [ ]:
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    !python -u scripts/train.py --streams rgb      --epochs 30 --batch 32 --workers 2 --cache /content/cache --amp
    !python -u scripts/train.py --streams forensic --epochs 30 --batch 32 --workers 2 --cache /content/cache --amp
    !python scripts/evaluate.py --ckpt checkpoints/rgb.pt --device cuda
    !python scripts/evaluate.py --ckpt checkpoints/forensic.pt --device cuda

In [ ]:
!python scripts/compare.py
display(IPImage('results/comparison.png', width=900))

## 10. Test your own images

In [ ]:
from google.colab import files
up = files.upload()   # pick one or more images
for name in up:
    !python analyze.py "{name}" --out reports
    display(IPImage(f"reports/{os.path.splitext(name)[0]}_report.png", width=1100))

In [ ]:
# or run on the bundled samples
!python analyze.py samples/splice.png samples/ai_generate.png samples/inpaint.png samples/authentic_1.png --out reports
display(IPImage('reports/splice_report.png', width=1100))

## 11. Launch the web app (Streamlit)
Option A opens the app inside Colab. If it shows a blank page, use Option B (public temporary link via Cloudflare – anyone with the link can open it while the cell runs).

In [ ]:
# Option A – Colab built-in proxy
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
import time; time.sleep(5)
from google.colab import output
output.serve_kernel_port_as_window(8501)

In [ ]:
# Option B – temporary public link
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!pgrep -f "streamlit run" > /dev/null || (nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &)
!nohup ./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1 &
import re; time.sleep(10)
print('Open:', re.findall(r'https://[-a-z0-9]+\.trycloudflare\.com', open('tunnel.log').read()))

## 12. Download the trained model + results
Copy `checkpoints/hybrid.pt` into the project's `checkpoints/` folder on your laptop – the app and `analyze.py` will pick it up automatically.

In [ ]:
from google.colab import files
!zip -q -r trained_outputs.zip checkpoints results reports results_training_curves.png
if SAVE_TO_DRIVE:
    !cp trained_outputs.zip "{DRIVE_DIR}/"
files.download('trained_outputs.zip')